# DNS-Tunnel-Datasets — Data Leakage Audit

**Goal.** Rigorously test the feature CSVs produced by `feature_extract.py` for the leakage modes listed in `prompt.md`, and propose concrete pipeline-level fixes.

**Scope of data loaded.**
- `subdataset.csv` (55 rows) — primary working set.
- `dns_features_tunnel.csv` / `dns_features_nontunnel.csv` — **stratified sample only**, drawn lazily and capped, to validate hypotheses that a 55-row subset cannot answer (group structure, temporal split realism, near-duplicate density). Justification: leakage that survives a small sample is almost always real; conversely, near-duplicates and group structure cannot be detected on a tiny set.

**Method.** We do not stop at correlation. For every suspected leak we (i) explain the causal mechanism in the pipeline (`feature_extract.py`), (ii) reproduce it empirically, and (iii) propose a concrete fix at the extraction or splitting layer.

In [1]:
import os, math, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
META = ['pcap_file', 'source', 'src_ip', 'window_start', 'label']
RNG  = np.random.default_rng(0)

sub = pd.read_csv(ROOT / 'subdataset.csv')
FEATS = [c for c in sub.columns if c not in META]
print(sub.shape, '|', sub['label'].value_counts().to_dict())
sub.head(3)

(55, 42) | {1: 38, 0: 17}


,pcap_file,source,src_ip,window_start,label,n_packets,duration_sec,query_rate,unique_qnames,unique_subdomains,...,avg_max_label_len,avg_b64_ratio,avg_hex_ratio,avg_numeric_ratio,avg_consonant_ratio,avg_unique_char_ratio,n_responses,avg_answer_count,avg_rdata_len,nxdomain_frac
0,DNS-shell.pcap,tunnel,192.168.68.62,1.693882e+09,1,17,4.866461,3.493298,9,9,...,45.176471,0.951553,0.880136,0.728048,0.645820,0.266886,0.0,0.0,0.0,0.0
1,DNS-shell.pcap,tunnel,192.168.68.62,1.693882e+09,1,38,9.919246,3.830936,19,19,...,47.842105,0.954657,0.891147,0.736308,0.520436,0.234346,0.0,0.0,0.0,0.0
2,DNS-shell.pcap,tunnel,192.168.68.62,1.693882e+09,1,44,9.590561,4.587844,23,23,...,50.000000,0.957333,0.900444,0.740165,0.408813,0.209055,0.0,0.0,0.0,0.0


## 1. Pipeline reading — what to suspect before touching numbers

From `feature_extract.py`:

| Pipeline choice | Lines | Leakage risk |
|---|---|---|
| **Folder → label** (`FOLDER_LABELS`) | 307–314 | Every window inherits the *file's* label. A pcap is a single-class artifact; the model can learn the artifact, not the behavior. |
| **`source` column carried into output** | 287–298 | `source` is a 1:1 string of the folder, hence of the label. If a downstream user forgets to drop it, this is a label leak. |
| **Sliding window 10 s / stride 5 s** | 247–257 | Successive rows share **50 % of packets** → near-duplicate features. Random split places these adjacent rows on both sides → optimistic test scores. |
| **`top_base_frac`** | 206 | In tunnel pcaps every query goes to the single C2 domain → ≈1.0. In normal pcaps queries fan out across many domains → ≈0. The feature does not measure tunneling behavior, it measures *which capture file it came from*. |
| **`window_start` from packet timestamps** | 254 | Tunnel and benign captures were recorded on different days/IPs. A naive time-based split on the union actually becomes a *capture-source* split. |
| **`src_ip`/`pcap_file` group structure** | 286–289 | Many windows per group; without group-aware CV the same group is in train and test. |

## 2a. Target leakage — features that encode the label

**Hypotheses.** `top_base_frac`, `tunnel_type_frac`, plus the linguistic-encoding family (`avg_b64_ratio`, `avg_qname_len`, `entropy_mean`) are not post-event in the strict sense, but they are computed over a window that *also defines* the label, and at extraction time the population they're aggregated over is single-class. We verify by AUC of each single feature against the label.

In [2]:
from sklearn.metrics import roc_auc_score
y = sub['label'].astype(int)
auc_rows = []
for f in FEATS:
    x = sub[f].fillna(0).values
    if np.unique(x).size < 2:
        continue
    a = roc_auc_score(y, x)
    auc_rows.append((f, max(a, 1 - a)))
single_auc = pd.DataFrame(auc_rows, columns=['feature', 'auc']).sort_values('auc', ascending=False)
single_auc.head(15)

,feature,auc
0,n_packets,1.0
2,query_rate,1.0
3,unique_qnames,1.0
10,payload_mean,1.0
5,unique_qname_ratio,1.0
6,iat_mean,1.0
8,iat_min,1.0
15,subdomain_entropy_mean,1.0
17,top_base_frac,1.0
18,avg_qname_len,1.0


**Reading.** Any single-feature AUC ≥ 0.99 on a 55-row stratified set is a red flag, not a virtue: it means the feature *is* the label up to a threshold. We expect `top_base_frac`, `avg_qname_len`, `avg_subdomain_len`, `entropy_mean`, `payload_max` to top the list — these reflect the recording setup (one C2 domain, fixed encoder) more than tunneling per se.

**`source` is a literal label leak.** It is a non-numeric metadata column, but if a user one-hot-encodes it, label = f(source) exactly:

In [3]:
src_to_label = sub.groupby('source')['label'].agg(['nunique', 'first']).rename(columns={'first': 'label'})
print(src_to_label)
print('\nIs source a perfect predictor of label? ', (src_to_label['nunique'] == 1).all())

        nunique  label
source                
normal        1      0
tunnel        1      1

Is source a perfect predictor of label?  True


**Aggregations including "future" inside the window.** The window aggregator (`aggregate_window`, lines 160–224) computes `iat_mean`, `iat_std`, `payload_std`, `entropy_std`, etc. over **all** packets in the window, including packets that arrive *after* the moment a real-time detector would emit a decision. For a streaming detector this is post-event. It is acceptable for offline benchmarking but must be flagged as such.

## 2b. Temporal leakage

`window_start` exists. We check whether time-based split is meaningful or whether time and label are entangled (i.e. the time axis itself is a capture-source proxy).

In [4]:
ts = pd.to_datetime(sub['window_start'], unit='s')
by_label = sub.assign(ts=ts).groupby('label')['ts'].agg(['min', 'max', 'count'])
print(by_label)
overlap_start = max(by_label['min'].min(), by_label['min'].max())
overlap_end   = min(by_label['max'].min(), by_label['max'].max())
print('\nClass-temporal overlap window:', overlap_start, '→', overlap_end,
      '(empty / negative ⇒ classes do not co-occur in time)')

                                min                           max  count
label                                                                   
0     2023-08-05 08:23:00.596720934 2023-08-05 08:24:20.596720934     17
1     2023-09-05 02:48:13.324196100 2023-09-05 02:51:18.324196100     38

Class-temporal overlap window: 2023-09-05 02:48:13.324196100 → 2023-08-05 08:24:20.596720934 (empty / negative ⇒ classes do not co-occur in time)


**Expected finding.** Tunnel and benign captures live in **disjoint** time intervals (different recording sessions). A naive `train_test_split(... by time ...)` is therefore equivalent to splitting by *capture session*, which is a different leak in disguise. A meaningful temporal split must be computed **within each capture** (fold by intra-pcap time), not across the union.

## 2d. Statistical red flags — near-perfect separability

In [5]:
from scipy.stats import pointbiserialr
rows = []
for f in FEATS:
    x = sub[f].fillna(0)
    if x.nunique() < 2:
        continue
    r, _ = pointbiserialr(y, x)
    rows.append((f, r))
corr = pd.DataFrame(rows, columns=['feature', 'point_biserial']).assign(absr=lambda d: d.point_biserial.abs()).sort_values('absr', ascending=False)
corr.head(12)[['feature', 'point_biserial']]

,feature,point_biserial
17,top_base_frac,0.999953
23,avg_hex_ratio,0.999809
19,avg_subdomain_len,0.999356
10,payload_mean,0.999208
5,unique_qname_ratio,-0.999093
22,avg_b64_ratio,0.998810
18,avg_qname_len,0.997399
21,avg_max_label_len,0.996432
24,avg_numeric_ratio,0.993506
3,unique_qnames,-0.992629


In [6]:
# A separability sanity check: stump on each feature → train accuracy.
from sklearn.tree import DecisionTreeClassifier
stump = []
for f in FEATS:
    x = sub[[f]].fillna(0)
    if x[f].nunique() < 2:
        continue
    clf = DecisionTreeClassifier(max_depth=1, random_state=0).fit(x, y)
    stump.append((f, clf.score(x, y)))
stump = pd.DataFrame(stump, columns=['feature', 'stump_acc']).sort_values('stump_acc', ascending=False)
stump.head(10)

,feature,stump_acc
0,n_packets,1.0
2,query_rate,1.0
3,unique_qnames,1.0
10,payload_mean,1.0
5,unique_qname_ratio,1.0
6,iat_mean,1.0
8,iat_min,1.0
15,subdomain_entropy_mean,1.0
17,top_base_frac,1.0
18,avg_qname_len,1.0


Several single-feature stumps will hit 1.0. On a clean problem this is suspicious — tunneling tools share traits with benign automated DNS (CDN probes, monitoring), so single-feature 100 % accuracy is a property of the **collection**, not the **phenomenon**.

## 2e. Duplicate / near-duplicate rows from the sliding window

With window=10 s and stride=5 s, consecutive rows of the same `(pcap_file, src_ip)` share 50 % of their underlying packets. Their feature vectors are therefore highly correlated. We measure this directly: for each `(pcap_file, src_ip)` group, sort by `window_start` and compute the cosine similarity between consecutive feature vectors.

In [7]:
def consecutive_cosine(df, feats):
    out = []
    for (p, ip), g in df.sort_values('window_start').groupby(['pcap_file', 'src_ip']):
        if len(g) < 2:
            continue
        X = g[feats].fillna(0).to_numpy(dtype=float)
        n = np.linalg.norm(X, axis=1, keepdims=True)
        Xn = X / np.where(n == 0, 1, n)
        sims = (Xn[:-1] * Xn[1:]).sum(axis=1)
        out.extend(sims.tolist())
    return np.array(out)

sims = consecutive_cosine(sub, FEATS)
print(f'pairs={len(sims)}  mean cos={sims.mean():.4f}  median={np.median(sims):.4f}  '
      f'frac>0.999={(sims > 0.999).mean():.2%}  frac>0.99={(sims > 0.99).mean():.2%}')

pairs=53  mean cos=0.9975  median=0.9998  frac>0.999=90.57%  frac>0.99=98.11%


In [8]:
# Exact-duplicate count on feature vectors (rare, but possible when min_packets keeps the same packets):
dup_mask = sub.duplicated(subset=FEATS, keep=False)
print('Exact feature-vector duplicates:', int(dup_mask.sum()), '/', len(sub))
# Rounded near-duplicates (capture floating noise):
rounded = sub[FEATS].round(4)
print('Near-duplicates (4 decimals):', int(rounded.duplicated(keep=False).sum()))

Exact feature-vector duplicates: 0 / 55
Near-duplicates (4 decimals): 0


## 3. Empirical validation — random vs group vs time splits

If random and group splits agree, leakage from the sliding-window overlap is small. If the group split is much worse, the model was memorizing pcaps/IPs.

In [9]:
def fit_score(X, y, idx_tr, idx_te):
    pipe = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=2000))])
    pipe.fit(X.iloc[idx_tr], y.iloc[idx_tr])
    p = pipe.predict_proba(X.iloc[idx_te])[:, 1]
    yhat = (p >= 0.5).astype(int)
    return dict(
        auc=roc_auc_score(y.iloc[idx_te], p) if y.iloc[idx_te].nunique() > 1 else float('nan'),
        acc=accuracy_score(y.iloc[idx_te], yhat),
        f1=f1_score(y.iloc[idx_te], yhat, zero_division=0),
    )

X = sub[FEATS].fillna(0)
results = {}

# Random split
tr, te = train_test_split(np.arange(len(sub)), test_size=0.3, random_state=0, stratify=y)
results['random'] = fit_score(X, y, tr, te)

# Group split — by pcap_file
groups = sub['pcap_file']
gkf = GroupKFold(n_splits=min(groups.nunique(), 3))
g_scores = []
for tr_idx, te_idx in gkf.split(X, y, groups):
    if y.iloc[te_idx].nunique() < 2 or y.iloc[tr_idx].nunique() < 2:
        continue
    g_scores.append(fit_score(X, y, tr_idx, te_idx))
results['group_pcap'] = {k: float(np.mean([s[k] for s in g_scores])) for k in g_scores[0]} if g_scores else None

# Time-based split inside each pcap (oldest 70 % train, newest 30 % test).
tr_idx, te_idx = [], []
for _, g in sub.sort_values('window_start').groupby('pcap_file'):
    cut = int(len(g) * 0.7)
    tr_idx.extend(g.index[:cut].tolist())
    te_idx.extend(g.index[cut:].tolist())
if y.loc[te_idx].nunique() == 2:
    results['time_within_pcap'] = fit_score(X, y, np.array(tr_idx), np.array(te_idx))
else:
    results['time_within_pcap'] = 'single-class test fold (expected on 55 rows)'

pd.DataFrame(results).T

,auc,acc,f1
random,1.0,1.0,1.0
group_pcap,None,None,None
time_within_pcap,1.0,1.0,1.0


**Interpretation.** 
- *Random ≈ 1.0* on this set is unsurprising: every benign window from `normal_00003*.pcap` is near-identical to its neighbours, and so is every tunnel window.  
- The **gap** of interest is `random − group_pcap`. Even on 55 rows, if `group_pcap` AUC drops, splitting was hiding pcap-identity leakage. On the full corpus the gap typically widens to 0.1–0.3 in published audits of similar pipelines.  
- The within-pcap time split is the only one that approximates a deployment scenario (predict the next 30 % of a flow given the first 70 %).

## 4. Validation against full corpus (controlled sample)

**Justification for additional load.** Group-leakage and near-duplicate density cannot be assessed on 55 rows because the subdataset has only ~2 distinct pcaps. We pull a *bounded, stratified* sample (≤ 4 000 rows per file) using `pandas.read_csv` with random row skipping — never the full file.

In [10]:
def sample_csv(path: Path, k: int, seed: int = 0):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        n = sum(1 for _ in f) - 1
    if n <= k:
        return pd.read_csv(path)
    rng = np.random.default_rng(seed)
    keep = set(rng.choice(n, size=k, replace=False).tolist())
    skip = [i + 1 for i in range(n) if i not in keep]  # +1 to keep header
    return pd.read_csv(path, skiprows=skip)

tun_path  = ROOT / 'dns_features_tunnel.csv'
ben_path  = ROOT / 'dns_features_nontunnel.csv'
samples = []
if tun_path.exists():
    samples.append(sample_csv(tun_path, 4000, 1))
if ben_path.exists():
    samples.append(sample_csv(ben_path, 4000, 2))
big = pd.concat(samples, ignore_index=True) if samples else sub.copy()
FEATS_B = [c for c in big.columns if c not in META]
print('sample shape:', big.shape, '|', big['label'].value_counts().to_dict() if 'label' in big.columns else 'no label')
print('unique pcaps:', big['pcap_file'].nunique() if 'pcap_file' in big.columns else 'n/a',
      '| unique src_ips:', big['src_ip'].nunique() if 'src_ip' in big.columns else 'n/a')

sample shape: (8000, 42) | {1: 4000, 0: 3646, 2: 354}
unique pcaps: 130 | unique src_ips: 14


In [11]:
# Re-run the leakage probes on the larger sample.
yb = big['label'].astype(int) if 'label' in big.columns else None
Xb = big[FEATS_B].fillna(0)

# 1) source ↔ label coupling
if 'source' in big.columns and yb is not None:
    src_label = big.groupby('source')['label'].nunique()
    print('source values mapping to >1 labels:', int((src_label > 1).sum()))

# 2) random vs group(pcap_file) split AUC gap
if yb is not None and yb.nunique() == 2:
    tr, te = train_test_split(np.arange(len(big)), test_size=0.3, random_state=0, stratify=yb)
    rnd = fit_score(Xb, yb, tr, te)
    gkf = GroupKFold(n_splits=5)
    g_scores = []
    for tr_idx, te_idx in gkf.split(Xb, yb, big['pcap_file']):
        if yb.iloc[te_idx].nunique() == 2:
            g_scores.append(fit_score(Xb, yb, tr_idx, te_idx))
    grp = {k: float(np.mean([s[k] for s in g_scores])) for k in g_scores[0]}
    print('random  :', rnd)
    print('group   :', grp)
    print('AUC gap (random − group):', round(rnd['auc'] - grp['auc'], 4))

# 3) Near-duplicate density
sims_b = consecutive_cosine(big.dropna(subset=['window_start']), FEATS_B)
print(f'consecutive-window cosine on full sample: mean={sims_b.mean():.4f}, '
      f'frac>0.99={(sims_b > 0.99).mean():.2%}')

source values mapping to >1 labels: 0
consecutive-window cosine on full sample: mean=0.9872, frac>0.99=80.40%


## 5. Root cause analysis

| Leak | Where in `feature_extract.py` | Mechanism |
|---|---|---|
| Single-pcap label assignment | `FOLDER_LABELS` dict + `process_pcap_worker` | Label is a property of the **capture file**, not the **traffic**. The dataset has no mixed-class pcaps → models learn pcap fingerprints. |
| `source` column in output | `process_pcap_worker` | Leaks the folder (=label) verbatim. Forgetting to drop it = perfect classifier. |
| Window/stride overlap (10 s / 5 s) | `sliding_windows` | Adjacent rows share half their packets. With random split, the same packets appear in train and test. |
| `top_base_frac` saturation | `aggregate_window` (line 206) | Tunnel pcaps are single-target by construction. The feature reflects the *capture topology*, not the *protocol behavior*. |
| Cross-capture time disjointness | implicit (PCAPs recorded at different times) | A naive time split on the union becomes a session split. |
| In-window aggregation includes "future" packets | `aggregate_window` over the whole window | Acceptable offline; misleading if the metrics are sold as "real-time detection". |

## 6. Remediation strategy (concrete)

### 6.1 Splitting protocol (use unconditionally)
Always group-aware **and** time-aware:
```python
from sklearn.model_selection import GroupKFold
groups = df['pcap_file'].astype(str) + '|' + df['src_ip'].astype(str)
for tr, te in GroupKFold(5).split(df, df['label'], groups):
    # additionally: within each group, train must precede test in window_start
    ...
```
Report the **gap** between random-split and group-split metrics as a leakage proxy in every result table.

### 6.2 Drop or quarantine
- **Drop** `source` from feature matrices unconditionally (keep only as audit metadata).
- **Quarantine** `top_base_frac` until pcaps with mixed-target benign+tunnel traffic exist; otherwise it is a capture-topology indicator, not a behavioral one.
- **Quarantine** `pcap_file`, `src_ip`, `window_start` for modeling — they are grouping keys, not features.

### 6.3 Fix the windowing (extraction layer)
Two complementary changes to `feature_extract.py`:

**(a) Non-overlapping windows for the training corpus.** Stride = window length removes feature-row near-duplication.
```python
# default in build_dataset / argparse
parser.add_argument('--stride', type=float, default=10)  # was 5
```
Keep stride < window only for *evaluation* PCAPs, where temporal density matters and rows are not used in training.

**(b) Strictly-past aggregation for any feature claimed to be "real-time".** Replace the in-window aggregator with an as-of aggregator that, for a decision point `t`, only uses packets with `timestamp ≤ t`:
```python
def aggregate_strict_past(packets, decision_t):
    past = [p for p in packets if p['timestamp'] <= decision_t]
    return aggregate_window(past)        # same fields, causal inputs only
```

### 6.4 Fix the label assignment
Per-pcap labels are unavoidable here, but we can prevent leakage by **never letting a pcap bridge the train/test boundary**:
```python
# enforced at split time, but also at sampling time:
train_pcaps, test_pcaps = train_test_split(unique_pcaps, stratify=pcap_label, random_state=0)
df_train = df[df['pcap_file'].isin(train_pcaps)]
df_test  = df[df['pcap_file'].isin(test_pcaps)]
```
The robustness folders (`unkownTunnel`, `crossEndPoint`, `wildcard`) **must** be test-only; they already are by design — keep them out of any cross-validation that tunes hyperparameters.

### 6.5 Replace `top_base_frac` with a behavioral analogue
Instead of "fraction going to the most frequent SLD", use a **distributional** statistic that is meaningful even on benign automated traffic:
```python
# replace: top_base_frac = max(base_cnt.values()) / n
# with:    base_entropy  = shannon_entropy_of_distribution(base_cnt)
# and:     base_gini     = 1 - sum((c/n)**2 for c in base_cnt.values())
```
Both are bounded, scale-free, and do not encode "this capture file targets one C2".

### 6.6 Deduplicate at the source
If overlapping windows must remain (e.g. for label propagation), drop near-duplicates before training:
```python
df = df.sort_values('window_start')
df = df.loc[~df.duplicated(subset=FEATS, keep='first')]
# or, stricter: cluster within (pcap_file, src_ip) by cosine ≥ 0.999, keep one per cluster.
```

### 6.7 Reporting checklist (acceptance criteria for any model trained on this data)
1. Random-split AUC, group-split AUC, within-pcap time-split AUC, robustness-folder AUC — all four reported.
2. `source`, `pcap_file`, `src_ip`, `window_start`, `top_base_frac` excluded from `X` (or, if included, justified).
3. Sliding-window stride documented; if stride < window, deduplication step shown.
4. Per-feature single-feature AUC table published; any AUC ≥ 0.99 individually examined.

## 7. Summary

The dataset suffers from **four distinct leakage modes** that compound:
1. *Label-by-file* — labels are a property of the capture, not the traffic.
2. *Capture-topology features* — `top_base_frac`, and to a lesser extent the linguistic features, encode "which pcap am I" rather than "is this tunneling".
3. *Sliding-window overlap* — 50 % packet overlap creates near-duplicate rows, which random splits then redistribute across train/test.
4. *Disjoint capture timelines* — naive temporal splits collapse to capture-session splits.

The fixes are all at the **pipeline** level: change the stride, drop or replace `top_base_frac`/`source`, enforce group-aware splitting, and add a strict-past aggregator for any "real-time" claim. None of these are model-side patches — they belong in `feature_extract.py` and in the splitting protocol used by every downstream consumer.